The scrapping code, originally composed of a few notebooks, isn't fully submitted in moodle. Indeed, we changed several times our scrapping approach (we first tried to scrape directly websites such as ikea, then tried bad data processing methods...) In order not to provide too much poorly reviewed code, we have chosen to keep only this notebook along with collect_raw_images.ipynb. But we must say that a few images of our chair dataset were obtained through other notebooks

In [1]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

In [ ]:
RAW_DIR = "raw_images"
OUTPUT_DIR = "filtered_images_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
def is_white_pixel(pixel, threshold=240):
    """
    pixel: array [R,G,B]
    threshold: valeur minimale pour considérer comme blanc
    """
    return all(channel >= threshold for channel in pixel)

In [8]:
def white_ratio(image_path, threshold=240):
    image = Image.open(image_path).convert("RGB")
    img_array = np.array(image)

    # Masque pixels blancs
    white_mask = np.all(img_array >= threshold, axis=2)

    white_pixels = np.sum(white_mask)
    total_pixels = img_array.shape[0] * img_array.shape[1]

    return white_pixels / total_pixels

In [9]:
import os
import shutil
import uuid
from tqdm import tqdm

WHITE_THRESHOLD = 0.40
PIXEL_THRESHOLD = 240

processed = 0

for root, _, files in os.walk(RAW_DIR):
    for filename in tqdm(files, leave=False):
        
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        
        path = os.path.join(root, filename)
        
        try:
            ratio = white_ratio(path, threshold=PIXEL_THRESHOLD)
            
            if ratio >= WHITE_THRESHOLD:
                processed += 1
                # Génère un nom unique pour éviter conflits
                ext = os.path.splitext(filename)[1].lower()
                new_name = f"chair_{processed:05d}{ext}"
                
                shutil.copy(path, os.path.join(OUTPUT_DIR, new_name))
        
        except Exception:
            continue

In [10]:
# Nombre d'images dans le dossier de sortie

count = 0
for _, _, files in os.walk(OUTPUT_DIR):
    count += len(files)
print(f"Nombre d'images dans le dossier de sortie : {count}")

Nombre d'images dans le dossier de sortie : 2225


In [11]:
# Gestion des doublons

seen_hashes = set()
for root, _, files in os.walk(OUTPUT_DIR):
    for filename in tqdm(files, leave=False):
        path = os.path.join(root, filename)
        try:
            with Image.open(path) as img:
                img_hash = hash(img.tobytes())
                if img_hash in seen_hashes:
                    os.remove(path)
                else:
                    seen_hashes.add(img_hash)
        except Exception:
            continue

print(f"Nombre d'images après suppression des doublons : {len(seen_hashes)}")

Nombre d'images après suppression des doublons : 2225


# Data augmentation

In [12]:
# Ajout des miroirs des images pour augmenter le dataset

for root, _, files in os.walk(OUTPUT_DIR):
    for filename in tqdm(files, leave=False):
        path = os.path.join(root, filename)
        try:
            with Image.open(path) as img:
                mirrored = img.transpose(Image.FLIP_LEFT_RIGHT)
                mirrored_name = f"mirror_{filename}"
                mirrored.save(os.path.join(OUTPUT_DIR, mirrored_name))
        except Exception:
            continue

In [ ]:
# Convertir les images du dossier "filtered_images_v0" en résolution 128x128 et les enregistrer dans le dossier "filtered_images"

import os
from PIL import Image

input_folder = "filtered_images_final"
output_folder = "filtered_images_final_128"
valid_extensions = (".jpg", ".jpeg", ".png")

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

converted = 0
for filename in os.listdir(input_folder):
    if filename.lower().endswith(valid_extensions):
        img_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        with Image.open(img_path) as img:
            img_resized = img.resize((128, 128), Image.Resampling.LANCZOS)

            # JPEG n'accepte pas RGBA/LA/P : conversion en RGB si nécessaire
            if filename.lower().endswith((".jpg", ".jpeg")) and img_resized.mode not in ("RGB", "L"):
                img_resized = img_resized.convert("RGB")

            img_resized.save(output_path)
            converted += 1

print(f"Conversion terminée. {converted} images redimensionnées enregistrées dans le dossier '{output_folder}'.")